<a href="https://colab.research.google.com/github/omerfra/AudioSeperator/blob/main/audiosep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Audio Stem Separation — Joint MusDB + MoisesDB

**DS25 Final Project — Technion**

This notebook trains a single 4-stem source-separation model (`vocals`,
`drums`, `bass`, `other`) on a **joint** dataset of MusDB18-HQ + MoisesDB.

## Why joint training (ML researcher view)

MusDB18-HQ is small (~100 train tracks) and stylistically narrow. MoisesDB
adds ~200 tracks with much greater instrumental diversity (per-track guitar,
piano, percussion, strings, winds, keys), but is more complicated to load:
each MoisesDB track has multiple sub-WAVs per stem folder, with ~11 possible
stem types per track (often with several silent).

Rather than treating MoisesDB as a separate model, we **collapse its
taxonomy into the same 4 MusDB stems** and train a single model on both. The
key ML decisions enabling this:

1. **Random per-stem cross-track remixing** — for a configurable fraction of
   training batches, each stem is independently sampled from a random track
   across *both* datasets, with random per-stem gain, then summed to form
   the mixture. This is the Demucs/Open-Unmix-style augmentation and gives
   combinatorial diversity (~tracks⁴ unique mixtures), not just track
   diversity.
2. **Stem-presence mask in the loss** — MoisesDB tracks frequently lack a
   stem entirely (no vocals on instrumentals, no bass on solo piano, …).
   The SI-SDR term is degenerate when the target is silent, so it is masked
   to non-silent stems. Mask/source/log terms still apply (predicting
   silence on a silent target is a valid learning signal).
3. **Per-subset and per-stem validation** — we log val loss and per-stem SDR
   separately for MusDB and MoisesDB each epoch, so catastrophic forgetting
   on either side is visible immediately.
4. **Full-state checkpointing** — model + optimizer + scheduler + scaler +
   epoch + history are saved so a disconnected Colab session resumes
   exactly where it left off.
5. **Warm-start by default** — `JOINT_WARM_START_FROM` points at the
   prior MusDB-only `best.pt`. Set to `None` to train from scratch.

## MoisesDB → 4-stem mapping

| MusDB output | MoisesDB stem folders summed (multi-WAV ⇒ summed within each folder) |
|---|---|
| `vocals` | `vocals` |
| `bass` | `bass` |
| `drums` | `drums`, `percussion` |
| `other` | `guitar`, `piano`, `other_keys`, `bowed_strings`, `wind`, `other_plucked`, `other` |

Missing folders → silence. Mapping is configurable in §3.


## 1. Setup

### 1.1 Install packages

In [ ]:
!pip install -q musdb stempeg librosa torch torchaudio numpy scipy matplotlib tqdm mir_eval soundfile noisereduce --quiet
!apt-get install -qq ffmpeg --quiet

### 1.2 Imports

In [ ]:
import os
import glob
import json
import hashlib
import random
from functools import lru_cache
from typing import Tuple, List, Dict, Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import autocast, GradScaler

import librosa
import librosa.display
import musdb
import soundfile as sf
import noisereduce as nr

import matplotlib.pyplot as plt
from IPython.display import Audio, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

# Colab-only mount; safe-skip outside Colab
try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not in Colab — drive mount skipped.')

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

### 1.3 Reproducibility and device

In [ ]:
def set_seed(seed=42, fast=True):
    """Reproducibility seed.

    `fast=True` enables cudnn.benchmark (autotunes conv algos for each input
    shape — gives a meaningful H100 speedup since our CQT input shape is
    fixed) and disables strict determinism. Set `fast=False` if exact
    bit-reproducibility matters more than throughput.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = not fast
        torch.backends.cudnn.benchmark     = fast


set_seed(42, fast=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# H100/A100 like TF32 for fp32 matmul paths (autocast already handles fp16/bf16).
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  {name} ({vram:.0f} GB, compute capability {cap[0]}.{cap[1]})')

## 2. Configuration

All hyperparameters, paths, and dataset constants in one place. Changing
sample rate / CQT params / stem count requires retraining from scratch.


In [ ]:
# ============================================
# AUDIO PARAMETERS (must match across the whole pipeline)
# ============================================
SAMPLE_RATE      = 44100
HOP_LENGTH       = 512
N_BINS           = 756
BINS_PER_OCTAVE  = 84
F_MIN            = 32.7
SEGMENT_LENGTH   = 10
SEGMENT_SAMPLES  = SAMPLE_RATE * SEGMENT_LENGTH

# ============================================
# MODEL PARAMETERS (architecture is frozen so warm-start works)
# ============================================
STEM_NAMES            = ['vocals', 'drums', 'bass', 'other']
N_STEMS               = len(STEM_NAMES)
ENCODER_CHANNELS      = [1, 32, 64, 128, 256, 512]
LSTM_LAYERS           = 3
LSTM_HIDDEN           = 512
TRANS_LAYERS          = 4
TRANS_HEADS           = 8
DECODER_TRANS_LEVELS  = [256, 128]
DROPOUT_RATE          = 0.15

# ============================================
# TRAINING PARAMETERS
# ============================================
BATCH_SIZE          = 16          # H100/A100; drop to 6–8 for T4/V100
NUM_EPOCHS          = 120
LEARNING_RATE       = 3.0e-5     # scaled ~linearly with batch from the BS=6 setting
MAX_LR              = 2.0e-4
WEIGHT_DECAY        = 5e-5
GRADIENT_CLIP       = 0.5
EARLY_STOP_PATIENCE = 25
NUM_WORKERS         = 12          # match Colab H100 CPU count; lower for T4/V100
PREFETCH_FACTOR     = 4           # batches each worker preloads (keeps the GPU fed)

# Mixed precision. H100 / A100 → 'bf16' (no GradScaler, wider dynamic range than fp16).
# Pre-Ampere (T4 / V100)        → 'fp16' (uses GradScaler).
# CPU or debugging              → 'fp32'.
PRECISION           = 'bf16'

# `torch.compile` gives ~1.2–1.5× on H100 for this model size, but adds
# ~30–60s warm-up on the first epoch. Set False if it fails on your graph.
USE_TORCH_COMPILE   = True

# ============================================
# LOSS WEIGHTS
# ============================================
MASK_LOSS_WEIGHT   = 1.0
SOURCE_LOSS_WEIGHT = 0.5
LOG_LOSS_WEIGHT    = 0.3
SDR_LOSS_WEIGHT    = 0.1
STEM_WEIGHTS       = {'vocals': 2.0, 'drums': 1.0, 'bass': 1.0, 'other': 0.8}

# Energy threshold below which a stem is considered silent — SDR loss
# is masked out on these samples (per-batch, per-stem). 1e-4 RMS ≈ -80 dBFS.
SILENCE_RMS_THRESHOLD = 1e-4

# ============================================
# DATA AUGMENTATION
# ============================================
USE_PITCH_SHIFT     = True
PITCH_SHIFT_RANGE   = (-2, 2)
PITCH_SHIFT_PROB    = 0.2          # cut down — pitch_shift is slow on CPU
USE_TIME_STRETCH    = True
TIME_STRETCH_RANGE  = (0.9, 1.1)
TIME_STRETCH_PROB   = 0.2
GAIN_RANGE          = (0.7, 1.3)   # global gain (applied uniformly to mix+stems)
PER_STEM_GAIN_RANGE = (0.25, 1.25) # independent per-stem gain (real and remix samples)

# ============================================
# JOINT TRAINING — KEY KNOBS
# ============================================
REMIX_PROB          = 0.5    # P(use cross-track remix) per training sample
                              # 0.0 = pure real-track training (Demucs baseline behavior)
                              # 1.0 = pure synthetic mixtures (max diversity, no real-context)
MUSDB_SAMPLER_WEIGHT  = 1.0   # relative weight per MusDB sample in the WeightedRandomSampler
MOISES_SAMPLER_WEIGHT = 1.0   # relative weight per MoisesDB sample; 1/1 = ~50/50 per epoch

# ============================================
# PATHS
# ============================================
MUSDB_PATH               = '/content/drive/MyDrive/Data/musdb18-hq'
MOISESDB_PATH            = '/content/drive/MyDrive/Data/moisedb'
CHECKPOINT_DIR           = '/content/drive/MyDrive/Data1/stem_separation_transformer/joint'
JOINT_WARM_START_FROM    = '/content/drive/MyDrive/Data1/stem_separation_transformer/best.pt'  # None = scratch
RESULTS_DIR              = './results'

# ============================================
# MOISESDB STEM COLLAPSE → 4-STEM TARGETS
# ============================================
# Each MoisesDB stem subfolder may contain multiple WAVs — they are summed
# to form that "stem". The mapping below then groups stems into the MusDB 4.
MOISESDB_STEM_GROUPS = {
    'vocals': ['vocals'],
    'drums':  ['drums', 'percussion'],
    'bass':   ['bass'],
    'other':  ['guitar', 'piano', 'other_keys', 'bowed_strings', 'wind', 'other_plucked', 'other'],
}
assert list(MOISESDB_STEM_GROUPS.keys()) == STEM_NAMES

MOISES_VAL_FRAC  = 0.10
MOISES_TEST_FRAC = 0.10
MOISES_SPLIT_SEED = 42

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('=' * 60)
print('CONFIGURATION')
print('=' * 60)
print(f'Stems              : {STEM_NAMES}')
print(f'Segment length     : {SEGMENT_LENGTH}s ({SEGMENT_SAMPLES} samples)')
print(f'Batch size         : {BATCH_SIZE}')
print(f'LR schedule        : {LEARNING_RATE} → {MAX_LR}')
print(f'Remix probability  : {REMIX_PROB}')
print(f'Warm start         : {JOINT_WARM_START_FROM or "(scratch)"}')
print(f'Checkpoint dir     : {CHECKPOINT_DIR}')
print('=' * 60)

## 3. Data Loading

We load two datasets that expose the **same per-track interface** so they can
be mixed downstream without conditionals:

- `MUSDB18Dataset` — wraps the `musdb` library
- `MoisesDBDataset` — walks `MOISESDB_PATH`, sums per-folder WAVs, collapses
  to the 4 MusDB stems

Both yield the same `__getitem__` dict so they're interchangeable in a
`ConcatDataset`. They also expose `load_track_audio(idx)` returning
`(mixture, [stem_audios], [stem_presence_flags])` so the **remix dataset**
in §3.4 can grab individual stems from arbitrary tracks across both
datasets.


### 3.1 Shared audio/CQT/augmentation helpers

In [ ]:
def compute_cqt(audio, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE, f_min=F_MIN):
    cqt = librosa.cqt(audio, sr=sr, hop_length=hop_length,
                      n_bins=n_bins, bins_per_octave=bins_per_octave, fmin=f_min)
    mag = np.abs(cqt).astype(np.float32)
    phase = np.angle(cqt).astype(np.float32)
    log_mag = np.log1p(mag).astype(np.float32)
    return log_mag, mag, phase


def extract_segment(audio, segment_samples, start=None, random_segments=True):
    total = len(audio)
    if total < segment_samples:
        return np.pad(audio, (0, segment_samples - total))
    if start is None:
        start = np.random.randint(0, total - segment_samples) if random_segments else 0
    return audio[start:start + segment_samples]


def apply_joint_augmentation(mix_audio, stem_audios, sr=SAMPLE_RATE,
                             segment_samples=SEGMENT_SAMPLES):
    """Augmentations applied jointly to mix and every stem.

    Pitch shift / time stretch must be applied identically to keep the
    invariant `mix == sum(stems)`. Global gain too. Per-stem random gain is
    handled separately (and re-sums the mix accordingly).
    """
    # Global gain — applied identically to mix and all stems
    gain = np.random.uniform(*GAIN_RANGE)
    mix_audio = mix_audio * gain
    stem_audios = [s * gain for s in stem_audios]

    if USE_PITCH_SHIFT and np.random.random() < PITCH_SHIFT_PROB:
        n_steps = np.random.uniform(*PITCH_SHIFT_RANGE)
        mix_audio = librosa.effects.pitch_shift(mix_audio, sr=sr, n_steps=n_steps)
        stem_audios = [librosa.effects.pitch_shift(s, sr=sr, n_steps=n_steps) for s in stem_audios]

    if USE_TIME_STRETCH and np.random.random() < TIME_STRETCH_PROB:
        rate = np.random.uniform(*TIME_STRETCH_RANGE)
        mix_audio = librosa.effects.time_stretch(mix_audio, rate=rate)
        stem_audios = [librosa.effects.time_stretch(s, rate=rate) for s in stem_audios]
        # Re-window to segment_samples after time stretch
        if len(mix_audio) > segment_samples:
            start = np.random.randint(0, len(mix_audio) - segment_samples)
            mix_audio = mix_audio[start:start + segment_samples]
            stem_audios = [s[start:start + segment_samples] for s in stem_audios]
        elif len(mix_audio) < segment_samples:
            mix_audio = np.pad(mix_audio, (0, segment_samples - len(mix_audio)))
            stem_audios = [np.pad(s, (0, segment_samples - len(s))) for s in stem_audios]

    return mix_audio, stem_audios


def build_sample_dict(mix_audio, stem_audios):
    """Convert raw waveforms to the standard dict used by the trainer."""
    mix_log_mag, mix_mag, mix_phase = compute_cqt(mix_audio)
    stem_log_mags, stem_mags = [], []
    for s in stem_audios:
        slm, sm, _ = compute_cqt(s)
        stem_log_mags.append(slm)
        stem_mags.append(sm)
    # Stem-presence flag: True iff the stem has non-trivial energy
    presence = np.array(
        [float(np.sqrt(np.mean(s ** 2)) > SILENCE_RMS_THRESHOLD) for s in stem_audios],
        dtype=np.float32,
    )
    return {
        'mixture':       torch.from_numpy(mix_log_mag).unsqueeze(0),
        'stems':         torch.from_numpy(np.stack(stem_log_mags)),
        'mixture_mag':   torch.from_numpy(mix_mag),
        'stems_mag':     torch.from_numpy(np.stack(stem_mags)),
        'mixture_phase': torch.from_numpy(mix_phase),
        'stems_audio':   torch.from_numpy(np.stack(stem_audios).astype(np.float32)),
        'mixture_audio': torch.from_numpy(mix_audio.astype(np.float32)),
        'stem_presence': torch.from_numpy(presence),  # (n_stems,) in {0,1}
    }


print('Shared helpers defined.')

### 3.2 MUSDB18Dataset

In [ ]:
class MUSDB18Dataset(Dataset):
    """MUSDB18-HQ wrapper with the standard interface used by the trainer
    and the remix dataset."""

    def __init__(self, mus_db, samples_per_track=8, random_segments=True,
                 augment=True, preload=True):
        self.tracks = list(mus_db.tracks)
        self.samples_per_track = samples_per_track
        self.random_segments = random_segments
        self.augment = augment
        self.stem_names = list(STEM_NAMES)

        self._cache = []
        if preload:
            print(f'Pre-loading {len(self.tracks)} MUSDB tracks...')
            for track in tqdm(self.tracks, desc='Load MUSDB'):
                mix = track.audio.mean(axis=1).astype(np.float32)
                stems = [track.targets[s].audio.mean(axis=1).astype(np.float32)
                         for s in self.stem_names]
                self._cache.append((mix, stems))
        else:
            self._cache = [None] * len(self.tracks)

    @property
    def num_tracks(self):
        return len(self.tracks)

    def load_track_audio(self, track_idx):
        """Return (mixture, [stem_audios], [presence_flags]) for one track."""
        cached = self._cache[track_idx]
        if cached is None:
            track = self.tracks[track_idx]
            mix = track.audio.mean(axis=1).astype(np.float32)
            stems = [track.targets[s].audio.mean(axis=1).astype(np.float32)
                     for s in self.stem_names]
            self._cache[track_idx] = (mix, stems)
        else:
            mix, stems = cached
        presence = [np.sqrt(np.mean(s ** 2)) > SILENCE_RMS_THRESHOLD for s in stems]
        return mix, stems, presence

    def __len__(self):
        return len(self.tracks) * self.samples_per_track

    def __getitem__(self, idx):
        track_idx = idx % len(self.tracks)
        mix, stems, _ = self.load_track_audio(track_idx)
        mix, stems = mix.copy(), [s.copy() for s in stems]

        total = len(mix)
        if self.random_segments and total > SEGMENT_SAMPLES:
            start = np.random.randint(0, total - SEGMENT_SAMPLES)
        else:
            sample_in_track = idx // len(self.tracks)
            start = min(sample_in_track * SEGMENT_SAMPLES, max(0, total - SEGMENT_SAMPLES))

        mix_seg = extract_segment(mix, SEGMENT_SAMPLES, start, self.random_segments)
        stem_segs = [extract_segment(s, SEGMENT_SAMPLES, start, self.random_segments) for s in stems]

        if self.augment:
            mix_seg, stem_segs = apply_joint_augmentation(mix_seg, stem_segs)

        return build_sample_dict(mix_seg, stem_segs)


print('MUSDB18Dataset defined.')

### 3.3 MoisesDBDataset

In [ ]:
class MoisesDBDataset(Dataset):
    """MoisesDB collapsed to the 4 MusDB stems.

    Each track folder contains one sub-folder per stem; each sub-folder
    contains one or more WAVs that must be **summed** to form the full stem.
    Missing sub-folders → silence.

    `preload=False` by default — MoisesDB is too large for full RAM preload
    on Colab. We use an LRU cache per track instead.
    """

    def __init__(self, root, track_ids, samples_per_track=8, random_segments=True,
                 augment=True, stem_groups=None, cache_size=8):
        self.root = root
        self.track_ids = list(track_ids)
        self.samples_per_track = samples_per_track
        self.random_segments = random_segments
        self.augment = augment
        self.stem_groups = stem_groups or MOISESDB_STEM_GROUPS
        self.stem_names = list(self.stem_groups.keys())
        # LRU-cache the (mix, stems) tuple per track id
        self._load_track_cached = lru_cache(maxsize=cache_size)(self._load_track_uncached)

    @property
    def num_tracks(self):
        return len(self.track_ids)

    def _sum_folder(self, folder):
        """Sum every *.wav in `folder`, resampling to SAMPLE_RATE."""
        if not os.path.isdir(folder):
            return None
        wavs = sorted(glob.glob(os.path.join(folder, '*.wav')))
        if not wavs:
            return None
        summed, max_len = None, 0
        loaded = []
        for w in wavs:
            try:
                audio, sr = sf.read(w, dtype='float32', always_2d=False)
            except Exception as e:
                print(f'  [skip] {w}: {e}')
                continue
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            if sr != SAMPLE_RATE:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
            audio = audio.astype(np.float32, copy=False)
            loaded.append(audio)
            max_len = max(max_len, len(audio))
        if not loaded:
            return None
        summed = np.zeros(max_len, dtype=np.float32)
        for a in loaded:
            if len(a) < max_len:
                a = np.pad(a, (0, max_len - len(a)))
            summed += a
        return summed

    def _load_track_uncached(self, track_id):
        track_dir = os.path.join(self.root, track_id)
        raw = {}
        max_len = 0
        for stem_name, folder_names in self.stem_groups.items():
            parts = []
            for fn in folder_names:
                part = self._sum_folder(os.path.join(track_dir, fn))
                if part is not None:
                    parts.append(part)
                    max_len = max(max_len, len(part))
            raw[stem_name] = parts
        if max_len == 0:
            raise RuntimeError(f'No audio for MoisesDB track {track_id}')
        stems = []
        for stem_name in self.stem_names:
            stem = np.zeros(max_len, dtype=np.float32)
            for p in raw[stem_name]:
                if len(p) < max_len:
                    p = np.pad(p, (0, max_len - len(p)))
                stem += p
            stems.append(stem)
        mixture = np.sum(np.stack(stems, axis=0), axis=0).astype(np.float32)
        return mixture, stems

    def load_track_audio(self, track_idx):
        mix, stems = self._load_track_cached(self.track_ids[track_idx])
        presence = [np.sqrt(np.mean(s ** 2)) > SILENCE_RMS_THRESHOLD for s in stems]
        return mix, stems, presence

    def __len__(self):
        return len(self.track_ids) * self.samples_per_track

    def __getitem__(self, idx):
        track_idx = idx % len(self.track_ids)
        mix, stems, _ = self.load_track_audio(track_idx)
        mix, stems = mix.copy(), [s.copy() for s in stems]

        total = len(mix)
        if self.random_segments and total > SEGMENT_SAMPLES:
            start = np.random.randint(0, total - SEGMENT_SAMPLES)
        else:
            sample_in_track = idx // len(self.track_ids)
            start = min(sample_in_track * SEGMENT_SAMPLES, max(0, total - SEGMENT_SAMPLES))

        mix_seg = extract_segment(mix, SEGMENT_SAMPLES, start, self.random_segments)
        stem_segs = [extract_segment(s, SEGMENT_SAMPLES, start, self.random_segments) for s in stems]

        if self.augment:
            mix_seg, stem_segs = apply_joint_augmentation(mix_seg, stem_segs)

        return build_sample_dict(mix_seg, stem_segs)


def split_moisesdb_track_ids(root, val_frac=MOISES_VAL_FRAC,
                              test_frac=MOISES_TEST_FRAC, seed=MOISES_SPLIT_SEED):
    """Deterministic hash-based train/val/test split."""
    if not os.path.isdir(root):
        raise FileNotFoundError(f'MoisesDB root not found: {root}')
    all_ids = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d)) and not d.startswith('.')
    )
    train, val, test = [], [], []
    for tid in all_ids:
        h = int(hashlib.md5(f'{seed}:{tid}'.encode()).hexdigest(), 16) / 16**32
        if h < test_frac:
            test.append(tid)
        elif h < test_frac + val_frac:
            val.append(tid)
        else:
            train.append(tid)
    return train, val, test


print('MoisesDBDataset defined.')

### 3.4 Cross-track per-stem remix dataset (key joint-training augmentation)

Each `__getitem__` independently samples **one stem** from a random track in
a **random underlying dataset**, applies a random per-stem gain, and sums the
four stems into a synthetic mixture. The model never sees the same
combination twice — for `T` tracks and 4 stems, the combinatorial space is
`T⁴` (~10¹⁰ for our setup) and the per-stem gains add a continuous axis.

Standard Demucs / Open-Unmix recipe. Critical for:
- forcing the model to learn **source-conditional** masks rather than
  memorising track-level co-occurrence patterns,
- robustness to per-stem loudness changes at inference,
- mitigating MoisesDB's frequent-silent-stem problem (silent stems are
  rarely picked because the bank only indexes tracks where the stem is
  present).


In [ ]:
class StemBank:
    """Per-stem index of (dataset, track_idx) over multiple source datasets.

    Only tracks whose stem is **present** (non-silent) at load time are
    indexed. This guarantees the remix dataset never picks an all-zero
    stem — every synthetic sample has all 4 stems non-trivially present,
    which is the most informative regime for the model.
    """

    def __init__(self, datasets, stem_names=STEM_NAMES, verbose=True):
        self.datasets = datasets
        self.stem_names = stem_names
        self.index = {s: [] for s in stem_names}
        if verbose:
            print('Building StemBank (presence-filtered)...')
        for ds in datasets:
            ds_name = type(ds).__name__
            counts = {s: 0 for s in stem_names}
            for tid in tqdm(range(ds.num_tracks), desc=f'  {ds_name}', leave=False):
                _, _, presence = ds.load_track_audio(tid)
                for s_idx, s_name in enumerate(stem_names):
                    if presence[s_idx]:
                        self.index[s_name].append((ds, tid))
                        counts[s_name] += 1
            if verbose:
                print(f'  {ds_name}: ' + ', '.join(f'{s}={c}' for s, c in counts.items()))
        empty = [s for s, lst in self.index.items() if not lst]
        if empty:
            raise RuntimeError(f'StemBank: no tracks contain stems {empty}')

    def sample_stem_segment(self, stem_name, segment_samples=SEGMENT_SAMPLES, rng=None):
        rng = rng or np.random
        ds, tid = self.index[stem_name][rng.randint(len(self.index[stem_name]))]
        _, stems, _ = ds.load_track_audio(tid)
        s_idx = self.stem_names.index(stem_name)
        return extract_segment(stems[s_idx].copy(), segment_samples, random_segments=True)


class RemixDataset(Dataset):
    """Yields synthetic mixtures by per-stem random sampling across the StemBank."""

    def __init__(self, stem_bank, length, augment=True,
                 per_stem_gain_range=PER_STEM_GAIN_RANGE):
        self.stem_bank = stem_bank
        self.length = length
        self.augment = augment
        self.per_stem_gain_range = per_stem_gain_range

    def __len__(self):
        return self.length

    def __getitem__(self, _):
        stem_segs = []
        for s_name in self.stem_bank.stem_names:
            seg = self.stem_bank.sample_stem_segment(s_name, SEGMENT_SAMPLES)
            gain = np.random.uniform(*self.per_stem_gain_range)
            stem_segs.append(seg * gain)
        mix = np.sum(np.stack(stem_segs, axis=0), axis=0).astype(np.float32)

        if self.augment:
            mix, stem_segs = apply_joint_augmentation(mix, stem_segs)

        return build_sample_dict(mix, stem_segs)


class JointTrainingDataset(Dataset):
    """Per-sample stochastic switch between (real-track) and (remix) sampling.

    Indices < len(real_concat) draw from the real-track ConcatDataset.
    With probability `remix_prob`, the sample is replaced by a synthetic
    mixture from the StemBank. This keeps the WeightedRandomSampler's
    dataset-balance semantics intact while injecting remix diversity on top.
    """

    def __init__(self, real_dataset, stem_bank, remix_prob=REMIX_PROB, augment=True):
        self.real_dataset = real_dataset
        self.stem_bank = stem_bank
        self.remix_prob = remix_prob
        self.remix_dataset = RemixDataset(stem_bank, length=1, augment=augment)

    def __len__(self):
        return len(self.real_dataset)

    def __getitem__(self, idx):
        if np.random.random() < self.remix_prob:
            return self.remix_dataset[0]
        return self.real_dataset[idx]


print('Remix dataset components defined.')

### 3.5 Load MUSDB18-HQ + MoisesDB, build splits

In [ ]:
print('Loading MUSDB18-HQ...')
mus_train_db = musdb.DB(root=MUSDB_PATH, subsets='train', split='train', is_wav=True)
mus_valid_db = musdb.DB(root=MUSDB_PATH, subsets='train', split='valid', is_wav=True)
mus_test_db  = musdb.DB(root=MUSDB_PATH, subsets='test', is_wav=True)
print(f'  Train: {len(mus_train_db.tracks)}  Valid: {len(mus_valid_db.tracks)}  Test: {len(mus_test_db.tracks)}')

print('\nSplitting MoisesDB...')
moises_train_ids, moises_val_ids, moises_test_ids = split_moisesdb_track_ids(MOISESDB_PATH)
print(f'  Train: {len(moises_train_ids)}  Valid: {len(moises_val_ids)}  Test: {len(moises_test_ids)}')

### 3.6 Instantiate datasets and the joint training pipeline

In [ ]:
# Real-track datasets
musdb_train  = MUSDB18Dataset(mus_train_db, samples_per_track=8, random_segments=True,  augment=True,  preload=True)
musdb_valid  = MUSDB18Dataset(mus_valid_db, samples_per_track=2, random_segments=False, augment=False, preload=True)
musdb_test   = MUSDB18Dataset(mus_test_db,  samples_per_track=2, random_segments=False, augment=False, preload=True)

moises_train = MoisesDBDataset(MOISESDB_PATH, moises_train_ids, samples_per_track=8,
                                random_segments=True,  augment=True)
moises_valid = MoisesDBDataset(MOISESDB_PATH, moises_val_ids,  samples_per_track=2,
                                random_segments=False, augment=False)
moises_test  = MoisesDBDataset(MOISESDB_PATH, moises_test_ids, samples_per_track=2,
                                random_segments=False, augment=False)

# StemBank from training-split tracks only (no leakage to val/test)
stem_bank = StemBank([musdb_train, moises_train])

# Real-track concat + per-sample remix injection
real_train = ConcatDataset([musdb_train, moises_train])
joint_train = JointTrainingDataset(real_train, stem_bank,
                                   remix_prob=REMIX_PROB, augment=True)

# Balanced weighted sampler: equal expected mass to MusDB and MoisesDB per epoch
weights = (
    [MUSDB_SAMPLER_WEIGHT  / max(1, len(musdb_train))]  * len(musdb_train) +
    [MOISES_SAMPLER_WEIGHT / max(1, len(moises_train))] * len(moises_train)
)
n_per_epoch = 2 * min(len(musdb_train), len(moises_train))
sampler = WeightedRandomSampler(weights=weights, num_samples=n_per_epoch, replacement=True)

# persistent_workers keeps the dataloader pool alive across epochs (no fork overhead).
# prefetch_factor=4 keeps several batches ready so the H100 doesn't wait on librosa CQT.
loader_kwargs = dict(
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
)

joint_train_loader = DataLoader(
    joint_train, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **loader_kwargs)

# Validation loaders are pure real-track (no remix), evaluated per-dataset
musdb_val_loader   = DataLoader(musdb_valid,  BATCH_SIZE, shuffle=False, **loader_kwargs)
moises_val_loader  = DataLoader(moises_valid, BATCH_SIZE, shuffle=False, **loader_kwargs)
musdb_test_loader  = DataLoader(musdb_test,   BATCH_SIZE, shuffle=False, **loader_kwargs)
moises_test_loader = DataLoader(moises_test,  BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f'Joint train: {len(joint_train)} (real) | {n_per_epoch} drawn per epoch | '
      f'{n_per_epoch // BATCH_SIZE} batches')
print(f'Val: MusDB={len(musdb_valid)}  MoisesDB={len(moises_valid)}')

## 4. Model

The architecture is **identical** to the MusDB-only model so the prior
`best.pt` warm-starts with no shape mismatch. U-Net encoder/decoder over
log-magnitude CQT, BiLSTM + Transformer at the bottleneck, 4 sigmoid mask
heads (one per stem).


### 4.1 BiLSTM block (temporal modelling)

In [ ]:
class BiLSTMBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=3, dropout=0.4):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim // 2,
                            num_layers=num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.output_proj = nn.Linear(hidden_dim, input_dim)
        self.layer_norm  = nn.LayerNorm(input_dim)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * H, W, C)
        residual = x
        x = self.input_proj(x)
        x, _ = self.lstm(x)
        x = self.output_proj(x)
        x = self.dropout(x)
        x = self.layer_norm(x + residual)
        return x.reshape(B, H, W, C).permute(0, 3, 1, 2)

### 4.2 Transformer blocks (global context)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=20000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        if seq_len > self.pe.size(1):
            pe = torch.zeros(seq_len, self.d_model, device=x.device)
            position = torch.arange(0, seq_len, dtype=torch.float, device=x.device).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, self.d_model, 2, device=x.device).float()
                                  * (-np.log(10000.0) / self.d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            x = x + pe.unsqueeze(0)
        else:
            x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x_n = self.norm1(x)
        attn, _ = self.self_attn(x_n, x_n, x_n)
        x = x + self.dropout(attn)
        x = x + self.ffn(self.norm2(x))
        return x


class BottleneckTransformer(nn.Module):
    def __init__(self, d_model, nhead=8, num_layers=4, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * H, W, C)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return x.reshape(B, H, W, C).permute(0, 3, 1, 2)


class DecoderTransformer(nn.Module):
    def __init__(self, channels, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        dim_feedforward = channels * 2
        self.pos_encoding = PositionalEncoding(channels, dropout=dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(channels, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H * W, C)
        residual = x
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x) + residual
        return x.reshape(B, H, W, C).permute(0, 3, 1, 2)

### 4.3 U-Net components

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.conv(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        features = self.conv(x)
        return features, self.pool(features)


class UNetEncoder(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.blocks = nn.ModuleList([
            EncoderBlock(channels[i], channels[i + 1]) for i in range(len(channels) - 1)])

    def forward(self, x):
        skips = []
        for block in self.blocks:
            feat, x = block(x)
            skips.append(feat)
        return x, skips


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, use_transformer=False, nhead=4):
        super().__init__()
        self.use_transformer = use_transformer
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)
        if use_transformer:
            self.transformer = DecoderTransformer(out_ch, nhead=nhead, num_layers=2)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            dh, dw = skip.shape[2] - x.shape[2], skip.shape[3] - x.shape[3]
            x = F.pad(x, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
        x = self.conv(torch.cat([x, skip], dim=1))
        if self.use_transformer:
            x = self.transformer(x)
        return x


class UNetDecoder(nn.Module):
    def __init__(self, decoder_channels, skip_channels, transformer_levels=None):
        super().__init__()
        transformer_levels = transformer_levels or []
        self.blocks = nn.ModuleList()
        for i in range(len(decoder_channels)):
            in_ch = decoder_channels[i]
            s_ch = skip_channels[i]
            out_ch = decoder_channels[i + 1] if i < len(decoder_channels) - 1 else s_ch
            use_trans = out_ch in transformer_levels
            self.blocks.append(DecoderBlock(in_ch, s_ch, out_ch, use_transformer=use_trans))

    def forward(self, x, skips):
        for block, skip in zip(self.blocks, reversed(skips)):
            x = block(x, skip)
        return x

### 4.4 StemSeparationNet (assembled)

In [ ]:
class StemSeparationNet(nn.Module):
    """U-Net + BiLSTM + Transformer producing N sigmoid stem masks."""

    def __init__(self, encoder_channels=ENCODER_CHANNELS, n_stems=N_STEMS,
                 dropout=DROPOUT_RATE, lstm_layers=LSTM_LAYERS, lstm_hidden=LSTM_HIDDEN,
                 trans_layers=TRANS_LAYERS, trans_heads=TRANS_HEADS,
                 decoder_trans_levels=DECODER_TRANS_LEVELS):
        super().__init__()
        self.n_stems = n_stems
        self.encoder = UNetEncoder(encoder_channels)
        bottleneck_ch = encoder_channels[-1]

        self.bilstm = BiLSTMBlock(input_dim=bottleneck_ch, hidden_dim=lstm_hidden,
                                  num_layers=lstm_layers, dropout=0.4)
        self.bottleneck_transformer = BottleneckTransformer(
            d_model=bottleneck_ch, nhead=trans_heads, num_layers=trans_layers,
            dim_feedforward=bottleneck_ch * 2, dropout=dropout)
        self.bottleneck_conv = ConvBlock(bottleneck_ch, bottleneck_ch)

        decoder_channels = list(reversed(encoder_channels[1:]))
        skip_channels    = list(reversed(encoder_channels[1:]))
        self.decoder = UNetDecoder(decoder_channels, skip_channels, decoder_trans_levels)

        decoder_out_ch = encoder_channels[1]
        self.final_up = nn.ConvTranspose2d(decoder_out_ch, decoder_out_ch, kernel_size=2, stride=2)
        self.stem_heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(decoder_out_ch, 32, 3, padding=1),
                nn.BatchNorm2d(32), nn.ReLU(inplace=True),
                nn.Conv2d(32, 1, 1), nn.Sigmoid())
            for _ in range(n_stems)
        ])

    def forward(self, x):
        original_shape = x.shape
        bottleneck, skips = self.encoder(x)
        bottleneck = self.bilstm(bottleneck)
        bottleneck = self.bottleneck_transformer(bottleneck)
        bottleneck = self.bottleneck_conv(bottleneck)
        decoded = self.decoder(bottleneck, skips)
        upsampled = self.final_up(decoded)
        if upsampled.shape[2:] != original_shape[2:]:
            upsampled = F.interpolate(upsampled, size=original_shape[2:],
                                       mode='bilinear', align_corners=False)
        return torch.cat([head(upsampled) for head in self.stem_heads], dim=1)


model = StemSeparationNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'StemSeparationNet: {n_params/1e6:.2f}M params')

with torch.no_grad():
    test_in = torch.randn(2, 1, N_BINS, 517).to(DEVICE)
    test_out = model(test_in)
    print(f'  Smoke test: in {tuple(test_in.shape)} → out {tuple(test_out.shape)}  '
          f'(range [{test_out.min():.3f}, {test_out.max():.3f}])')

## 5. Loss Function

Same multi-objective recipe as the MusDB stage (Wiener-mask MSE + linear
source L1 + log-magnitude L1 + spectral SI-SDR proxy), **with one critical
change** for joint training:

> The SDR term is masked to non-silent stems per sample.

Without the mask, MoisesDB samples with silent stems (e.g. an instrumental
track has `vocals_rms ≈ 0`) make `target_energy` ≈ ε, causing
`-10·log10(target/error)` to blow up to large negative values and dominate
the loss. The mask/source/log terms still apply on those silent stems —
those gradients correctly push the model to output silence.


In [ ]:
class SeparationLoss(nn.Module):
    def __init__(self, mask_weight=MASK_LOSS_WEIGHT, source_weight=SOURCE_LOSS_WEIGHT,
                 log_weight=LOG_LOSS_WEIGHT, sdr_weight=SDR_LOSS_WEIGHT,
                 stem_weights=None, stem_names=STEM_NAMES, eps=1e-8):
        super().__init__()
        self.mask_weight = mask_weight
        self.source_weight = source_weight
        self.log_weight = log_weight
        self.sdr_weight = sdr_weight
        self.stem_names = stem_names
        self.stem_weights = stem_weights or {s: 1.0 for s in stem_names}
        self.weight_tensor = torch.tensor([self.stem_weights[s] for s in stem_names])
        self.eps = eps

    def forward(self, pred_masks, mixture, target_stems, stem_presence=None, **kwargs):
        """pred_masks/mixture/target_stems are log-magnitude CQT-domain.
        stem_presence: (B, n_stems) in {0,1}. If None, all present."""
        mix_linear = torch.expm1(mixture.clamp(min=0))
        stems_linear = torch.expm1(target_stems.clamp(min=0))
        mix_linear_exp = mix_linear.expand_as(stems_linear)

        sum_stems = stems_linear.sum(dim=1, keepdim=True) + self.eps
        ideal_masks = stems_linear / sum_stems

        weights = self.weight_tensor.to(pred_masks.device).view(1, -1, 1, 1)
        if stem_presence is None:
            presence = torch.ones(pred_masks.size(0), pred_masks.size(1),
                                  device=pred_masks.device)
        else:
            presence = stem_presence.to(pred_masks.device).float()

        # Mask / source / log losses apply to ALL stems (silent or not).
        # Predicting silence on a silent target is a useful learning signal.
        mask_diff   = ((pred_masks - ideal_masks) ** 2) * weights
        mask_loss   = mask_diff.mean()
        pred_sources = pred_masks * mix_linear_exp
        source_loss = (torch.abs(pred_sources - stems_linear) * weights).mean()
        log_loss    = (torch.abs(torch.log1p(pred_sources) - target_stems) * weights).mean()

        total_loss = self.mask_weight   * mask_loss \
                   + self.source_weight * source_loss \
                   + self.log_weight    * log_loss

        # SI-SDR proxy — masked to non-silent stems only
        sdr_val = 0.0
        if self.sdr_weight > 0:
            target_energy = (stems_linear ** 2).sum(dim=(2, 3)) + self.eps   # (B, n_stems)
            error_energy  = ((pred_sources - stems_linear) ** 2).sum(dim=(2, 3)) + self.eps
            spectral_sdr  = 10 * torch.log10(target_energy / error_energy)    # (B, n_stems)
            stem_w = self.weight_tensor.to(spectral_sdr.device).unsqueeze(0)  # (1, n_stems)
            sdr_weighted = spectral_sdr * stem_w * presence
            n_active = presence.sum().clamp(min=1.0)
            mean_sdr = sdr_weighted.sum() / n_active
            sdr_loss = -mean_sdr / 20.0
            if not torch.isnan(sdr_loss) and not torch.isinf(sdr_loss):
                total_loss = total_loss + self.sdr_weight * sdr_loss
                sdr_val = mean_sdr.item()

        # Per-stem SDR breakdown for logging (NaN on stems with no presence)
        per_stem_sdr = {}
        if self.sdr_weight > 0:
            for i, s_name in enumerate(self.stem_names):
                pres_i = presence[:, i]
                if pres_i.sum() > 0:
                    per_stem_sdr[s_name] = (spectral_sdr[:, i] * pres_i).sum().item() / pres_i.sum().item()
                else:
                    per_stem_sdr[s_name] = float('nan')

        return total_loss, {
            'total':  total_loss.item(),
            'mask':   mask_loss.item(),
            'source': source_loss.item(),
            'log':    log_loss.item(),
            'sdr':    sdr_val,
            'per_stem_sdr': per_stem_sdr,
        }


criterion = SeparationLoss()
print(f'SeparationLoss: mask={MASK_LOSS_WEIGHT}, source={SOURCE_LOSS_WEIGHT}, '
      f'log={LOG_LOSS_WEIGHT}, sdr={SDR_LOSS_WEIGHT}')
print(f'Stem weights: {STEM_WEIGHTS}')

## 6. Training

`Trainer` supports:

- **Full-state checkpointing**: `last.pt` (every epoch) holds model +
  optimizer + scheduler + scaler + epoch + history + best_loss. `best.pt`
  is the best-val-loss snapshot.
- **Resume**: `Trainer.fit(resume=True)` picks up from `last.pt` exactly
  — useful when Colab disconnects.
- **Per-subset validation**: pass `val_loaders={'musdb': ..., 'moises': ...}`
  to log each one separately. Combined loss (mean of subsets) drives early
  stopping and `best.pt` selection — so we can't silently abandon one
  dataset.
- **Per-stem SDR logging**: per-stem SI-SDR is accumulated over non-silent
  stems each validation pass.


In [ ]:
_PRECISION_DTYPE = {'fp16': torch.float16, 'bf16': torch.bfloat16, 'fp32': None}


class Trainer:
    def __init__(self, model, criterion, optimizer, scheduler, device, checkpoint_dir,
                 grad_clip=1.0, patience=20, val_loaders=None, precision='bf16'):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.grad_clip = grad_clip
        self.patience = patience
        self.val_loaders = val_loaders or {}

        # Mixed-precision setup. bf16 has wider exponent range than fp16 and
        # does not need loss scaling — recommended on H100 / A100.
        self.amp_dtype = _PRECISION_DTYPE[precision]
        self.use_amp = self.amp_dtype is not None and device.type == 'cuda'
        self.use_scaler = self.use_amp and self.amp_dtype == torch.float16
        self.scaler = GradScaler() if self.use_scaler else None
        print(f'Trainer precision: {precision}  (amp={self.use_amp}, scaler={self.use_scaler})')

        self.history = {'train_loss': [], 'val_loss': [], 'lr': []}
        for name in self.val_loaders:
            self.history[f'val_loss_{name}'] = []
            self.history[f'val_sdr_{name}']  = []
            for s in STEM_NAMES:
                self.history[f'val_sdr_{name}_{s}'] = []
        self.best_loss = float('inf')
        self.no_improve = 0
        self.start_epoch = 0

    # ---- training/validation primitives ----
    def _forward_loss(self, batch):
        mix = batch['mixture'].to(self.device, non_blocking=True)
        stems = batch['stems'].to(self.device, non_blocking=True)
        presence = batch.get('stem_presence')
        if presence is not None:
            presence = presence.to(self.device, non_blocking=True)
        masks = self.model(mix)
        return self.criterion(masks, mix, stems, stem_presence=presence)

    def _autocast(self):
        """Context manager wrapping the active autocast (or nullcontext for fp32)."""
        if self.use_amp:
            return autocast(device_type='cuda', dtype=self.amp_dtype)
        from contextlib import nullcontext
        return nullcontext()

    def train_epoch(self, loader):
        self.model.train()
        totals = {'loss': 0.0, 'n': 0}
        for batch in tqdm(loader, desc='train', leave=False):
            self.optimizer.zero_grad(set_to_none=True)
            with self._autocast():
                loss, d = self._forward_loss(batch)
            if self.use_scaler:
                self.scaler.scale(loss).backward()
                if self.grad_clip > 0:
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                if self.grad_clip > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()
            totals['loss'] += d['total']
            totals['n']    += 1
        return {'loss': totals['loss'] / max(totals['n'], 1)}

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        loss_sum, sdr_sum, n = 0.0, 0.0, 0
        per_stem_sum = {s: 0.0 for s in STEM_NAMES}
        per_stem_count = {s: 0 for s in STEM_NAMES}
        for batch in tqdm(loader, desc='val', leave=False):
            with self._autocast():
                _, d = self._forward_loss(batch)
            loss_sum += d['total']
            sdr_sum  += d['sdr']
            n        += 1
            for s, v in d['per_stem_sdr'].items():
                if not np.isnan(v):
                    per_stem_sum[s]   += v
                    per_stem_count[s] += 1
        per_stem_sdr = {s: (per_stem_sum[s] / per_stem_count[s]) if per_stem_count[s] else float('nan')
                        for s in STEM_NAMES}
        return {
            'loss': loss_sum / max(n, 1),
            'sdr':  sdr_sum  / max(n, 1),
            'per_stem_sdr': per_stem_sdr,
        }

    # ---- checkpointing ----
    def save_checkpoint(self, path, epoch):
        # If the model was wrapped by torch.compile, unwrap before saving so the
        # checkpoint loads cleanly into a fresh, uncompiled model later.
        raw_model = getattr(self.model, '_orig_mod', self.model)
        state = {
            'epoch': epoch,
            'model':     raw_model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict() if self.scheduler else None,
            'scaler':    self.scaler.state_dict() if self.scaler else None,
            'history':   self.history,
            'best_loss': self.best_loss,
            'no_improve': self.no_improve,
        }
        torch.save(state, path)

    def load_checkpoint(self, path):
        state = torch.load(path, map_location=self.device)
        raw_model = getattr(self.model, '_orig_mod', self.model)
        raw_model.load_state_dict(state['model'])
        self.optimizer.load_state_dict(state['optimizer'])
        if self.scheduler and state.get('scheduler'):
            self.scheduler.load_state_dict(state['scheduler'])
        if self.scaler and state.get('scaler'):
            self.scaler.load_state_dict(state['scaler'])
        self.history    = state.get('history', self.history)
        self.best_loss  = state.get('best_loss', float('inf'))
        self.no_improve = state.get('no_improve', 0)
        self.start_epoch = state.get('epoch', 0) + 1
        print(f'Resumed from epoch {self.start_epoch} (best_loss={self.best_loss:.4f})')

    # ---- main fit loop ----
    def fit(self, train_loader, epochs, resume=False):
        import time
        last_path = os.path.join(self.checkpoint_dir, 'last.pt')
        best_path = os.path.join(self.checkpoint_dir, 'best.pt')
        if resume and os.path.exists(last_path):
            self.load_checkpoint(last_path)

        print(f'\n{"="*60}\nTRAINING (start epoch {self.start_epoch+1})\n{"="*60}')
        for epoch in range(self.start_epoch, epochs):
            t0 = time.time()
            lr = self.optimizer.param_groups[0]['lr']
            train_m = self.train_epoch(train_loader)

            per_subset = {}
            for name, loader in self.val_loaders.items():
                m = self.validate(loader)
                per_subset[name] = m
                self.history[f'val_loss_{name}'].append(m['loss'])
                self.history[f'val_sdr_{name}'].append(m['sdr'])
                for s in STEM_NAMES:
                    self.history[f'val_sdr_{name}_{s}'].append(m['per_stem_sdr'][s])

            combined_loss = float(np.mean([m['loss'] for m in per_subset.values()])) \
                if per_subset else float('inf')
            combined_sdr  = float(np.mean([m['sdr']  for m in per_subset.values()])) \
                if per_subset else 0.0

            if self.scheduler:
                self.scheduler.step()

            self.history['train_loss'].append(train_m['loss'])
            self.history['val_loss'].append(combined_loss)
            self.history['lr'].append(lr)

            is_best = combined_loss < self.best_loss
            if is_best:
                self.best_loss = combined_loss
                self.no_improve = 0
                raw_model = getattr(self.model, '_orig_mod', self.model)
                torch.save({'model': raw_model.state_dict()}, best_path)
            else:
                self.no_improve += 1

            self.save_checkpoint(last_path, epoch)

            subset_str = ' | '.join(
                f'{n}: L={m["loss"]:.4f} SDR={m["sdr"]:.1f}'
                for n, m in per_subset.items())
            stem_str = ' | '.join(
                f'{s[:3]}={np.nanmean([per_subset[n]["per_stem_sdr"][s] for n in per_subset]):.1f}'
                for s in STEM_NAMES)
            print(f'Ep {epoch+1:3d}/{epochs} | tr={train_m["loss"]:.4f} | {subset_str} | '
                  f'stems[{stem_str}] | LR={lr:.1e} | {time.time()-t0:.0f}s '
                  f'{"★" if is_best else ""}')

            if self.no_improve >= self.patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
        return self.history


print('Trainer defined.')

### 6.1 Warm-start, optimizer, scheduler

In [ ]:
# Warm-start from prior MusDB-only checkpoint, if available.
if JOINT_WARM_START_FROM and os.path.exists(JOINT_WARM_START_FROM):
    print(f'Warm-starting from {JOINT_WARM_START_FROM}')
    ckpt = torch.load(JOINT_WARM_START_FROM, map_location=DEVICE)
    state_dict = ckpt.get('model', ckpt)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    print(f'  missing={len(missing)}, unexpected={len(unexpected)}')
else:
    print('Training from scratch (no warm-start).')

# Optional torch.compile — ~1.2–1.5× on H100. Wrap AFTER warm-start so the
# state_dict load uses the original parameter names.
if USE_TORCH_COMPILE and DEVICE.type == 'cuda':
    try:
        model = torch.compile(model, mode='default')
        print('torch.compile: enabled')
    except Exception as e:
        print(f'torch.compile failed ({e}); continuing uncompiled')

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)

trainer = Trainer(
    model, criterion, optimizer, scheduler, DEVICE, CHECKPOINT_DIR,
    grad_clip=GRADIENT_CLIP, patience=EARLY_STOP_PATIENCE,
    val_loaders={'musdb': musdb_val_loader, 'moises': moises_val_loader},
    precision=PRECISION,
)

### 6.2 Launch (or resume) training

Set `RESUME=True` to continue from the last saved epoch — the `Trainer`
restores model + optimizer + scheduler + scaler + history.


In [ ]:
RESUME = False
history = trainer.fit(joint_train_loader, NUM_EPOCHS, resume=RESUME)

## 7. Inference

Overlap-add reconstruction in the time domain. Phase is taken from the
mixture (standard mask-based separation).

In [ ]:
def separate_audio(model, audio, segment_length=SEGMENT_LENGTH, device=DEVICE):
    """Separate audio into the 4 stems using overlap-add."""
    model.eval()
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    segment_samples = int(segment_length * SAMPLE_RATE)
    total_samples = len(audio)
    overlap = segment_samples // 2
    hop = segment_samples - overlap
    window = np.hanning(segment_samples).astype(np.float32)

    out = {s: np.zeros(total_samples, dtype=np.float32) for s in STEM_NAMES}
    norm = np.zeros(total_samples, dtype=np.float32)
    padded = np.pad(audio, (overlap, segment_samples))
    positions = list(range(0, total_samples + overlap, hop))

    for start in tqdm(positions, desc='Separating'):
        seg = padded[start:start + segment_samples]
        if len(seg) < segment_samples:
            seg = np.pad(seg, (0, segment_samples - len(seg)))

        cqt = librosa.cqt(seg, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                          n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE, fmin=F_MIN)
        mag, phase = np.abs(cqt), np.angle(cqt)
        log_mag = np.log1p(mag)
        mag_tensor = torch.from_numpy(log_mag).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            if device.type == 'cuda':
                with autocast(device_type='cuda'):
                    masks = model(mag_tensor)
            else:
                masks = model(mag_tensor)
        masks = masks.cpu().numpy()[0]

        for i, s in enumerate(STEM_NAMES):
            est_mag = masks[i] * mag
            est_cqt = est_mag * np.exp(1j * phase)
            stem_audio = librosa.icqt(est_cqt, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                                       bins_per_octave=BINS_PER_OCTAVE, fmin=F_MIN)
            if len(stem_audio) < segment_samples:
                stem_audio = np.pad(stem_audio, (0, segment_samples - len(stem_audio)))
            stem_audio = stem_audio[:segment_samples]
            windowed = stem_audio * window

            os_, oe = start - overlap, start - overlap + segment_samples
            ss, se = max(0, -os_), segment_samples - max(0, oe - total_samples)
            ds, de = max(0, os_), min(total_samples, oe)
            if de > ds:
                out[s][ds:de] += windowed[ss:se]
        os_, oe = start - overlap, start - overlap + segment_samples
        ss, se = max(0, -os_), segment_samples - max(0, oe - total_samples)
        ds, de = max(0, os_), min(total_samples, oe)
        if de > ds:
            norm[ds:de] += window[ss:se]

    norm = np.maximum(norm, 1e-8)
    for s in STEM_NAMES:
        out[s] /= norm
    return out


def denoise_stem(audio, strength=0.8):
    return nr.reduce_noise(
        y=audio, sr=SAMPLE_RATE, stationary=False,
        prop_decrease=strength, n_fft=2048, hop_length=512,
        n_std_thresh_stationary=1.5,
    )


DENOISE_CONFIG = {'vocals': 0.95, 'drums': 0.5, 'bass': 0.6, 'other': 0.8}
print('Inference functions defined.')

## 8. Evaluation

Per-stem SDR on held-out splits of both datasets. Silent stems are
**excluded** from the SDR average (otherwise the metric is degenerate).

In [ ]:
def waveform_sdr(estimate, reference, eps=1e-8):
    """Standard SDR in the time domain (dB), scalar."""
    ref_energy = np.sum(reference ** 2) + eps
    err_energy = np.sum((estimate - reference) ** 2) + eps
    return 10.0 * np.log10(ref_energy / err_energy)


def evaluate_dataset_sdr(model, dataset, name, max_tracks=None, segment_length=SEGMENT_LENGTH):
    """Run full-track inference and accumulate per-stem SDR."""
    n_tracks = min(dataset.num_tracks, max_tracks) if max_tracks else dataset.num_tracks
    per_stem = {s: [] for s in STEM_NAMES}
    for tid in tqdm(range(n_tracks), desc=f'eval {name}'):
        mix, stems, presence = dataset.load_track_audio(tid)
        sep = separate_audio(model, mix, segment_length=segment_length)
        for i, s in enumerate(STEM_NAMES):
            if not presence[i]:
                continue
            est = sep[s][:len(stems[i])]
            ref = stems[i][:len(est)]
            per_stem[s].append(waveform_sdr(est, ref))
    summary = {s: (float(np.mean(per_stem[s])) if per_stem[s] else float('nan'))
               for s in STEM_NAMES}
    summary['overall'] = float(np.mean([v for v in summary.values() if not np.isnan(v)]))
    return summary, per_stem


def load_model_from(path):
    m = StemSeparationNet().to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE)
    state_dict = ckpt.get('model', ckpt)
    m.load_state_dict(state_dict)
    m.eval()
    return m


print('Evaluation helpers defined.')

### 8.1 Compare baseline (MusDB-only) vs joint checkpoint

In [ ]:
# Load the best joint checkpoint and (if available) the prior MusDB-only baseline.
joint_model = load_model_from(os.path.join(CHECKPOINT_DIR, 'best.pt'))

baseline_model = None
if JOINT_WARM_START_FROM and os.path.exists(JOINT_WARM_START_FROM):
    baseline_model = load_model_from(JOINT_WARM_START_FROM)

# Evaluate on small subsets first; widen with `max_tracks` once it's working.
EVAL_MAX_TRACKS = 5  # bump to None for full eval

print('\n== Joint model ==')
joint_musdb,  _ = evaluate_dataset_sdr(joint_model, musdb_test,  'MusDB',   EVAL_MAX_TRACKS)
joint_moises, _ = evaluate_dataset_sdr(joint_model, moises_test, 'MoisesDB', EVAL_MAX_TRACKS)

print('\n  MusDB SDR :', {k: f'{v:.2f}' for k, v in joint_musdb.items()})
print('  MoisesDB SDR:', {k: f'{v:.2f}' for k, v in joint_moises.items()})

if baseline_model is not None:
    print('\n== Baseline (MusDB-only) ==')
    base_musdb,  _ = evaluate_dataset_sdr(baseline_model, musdb_test,  'MusDB',   EVAL_MAX_TRACKS)
    base_moises, _ = evaluate_dataset_sdr(baseline_model, moises_test, 'MoisesDB', EVAL_MAX_TRACKS)
    print('\n  MusDB SDR :', {k: f'{v:.2f}' for k, v in base_musdb.items()})
    print('  MoisesDB SDR:', {k: f'{v:.2f}' for k, v in base_moises.items()})

    print('\n== Δ (joint − baseline), dB ==')
    print('  MusDB :', {k: f'{joint_musdb[k]-base_musdb[k]:+.2f}' for k in joint_musdb})
    print('  MoisesDB:', {k: f'{joint_moises[k]-base_moises[k]:+.2f}' for k in joint_moises})

### 8.2 Training-history plot

In [ ]:
def plot_history(history, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(history['train_loss'], label='train')
    for name in ['musdb', 'moises']:
        key = f'val_loss_{name}'
        if key in history and history[key]:
            axes[0].plot(history[key], label=f'val ({name})')
    axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    for name in ['musdb', 'moises']:
        key = f'val_sdr_{name}'
        if key in history and history[key]:
            axes[1].plot(history[key], label=f'val ({name})')
    axes[1].set_title('Validation SDR (combined per stem-weighting)')
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('dB'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    for s in STEM_NAMES:
        avg = []
        for ep in range(len(history.get('val_loss', []))):
            vals = [history[f'val_sdr_{name}_{s}'][ep]
                    for name in ['musdb', 'moises']
                    if f'val_sdr_{name}_{s}' in history and ep < len(history[f'val_sdr_{name}_{s}'])]
            vals = [v for v in vals if not np.isnan(v)]
            avg.append(np.mean(vals) if vals else np.nan)
        axes[2].plot(avg, label=s)
    axes[2].set_title('Per-stem val SDR (combined)')
    axes[2].set_xlabel('epoch'); axes[2].set_ylabel('dB'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()


# Load most recent training history (works mid-training too)
last_path = os.path.join(CHECKPOINT_DIR, 'last.pt')
if os.path.exists(last_path):
    plot_history(torch.load(last_path, map_location='cpu')['history'],
                 save_path=os.path.join(RESULTS_DIR, 'history.png'))

## 9. Try it yourself

Upload a WAV, run separation with the joint-trained checkpoint, listen.


In [ ]:
if IN_COLAB:
    print('Upload a WAV file:')
    uploaded = files.upload()
    if uploaded:
        user_filename = next(iter(uploaded))
        user_audio, sr = sf.read(user_filename, dtype='float32', always_2d=False)
        if user_audio.ndim > 1:
            user_audio = user_audio.mean(axis=1)
        if sr != SAMPLE_RATE:
            user_audio = librosa.resample(user_audio, orig_sr=sr, target_sr=SAMPLE_RATE)
        print(f'\nLoaded: {user_audio.shape} @ {SAMPLE_RATE}Hz')

        sep = separate_audio(joint_model, user_audio)

        print('\nOriginal mix:')
        display(Audio(user_audio, rate=SAMPLE_RATE))
        for s in STEM_NAMES:
            print(f'\nSeparated {s}:')
            display(Audio(sep[s], rate=SAMPLE_RATE))

            cleaned = denoise_stem(sep[s], strength=DENOISE_CONFIG[s])
            print(f'  (denoised, strength={DENOISE_CONFIG[s]}):')
            display(Audio(cleaned, rate=SAMPLE_RATE))
else:
    print('Skipping upload UI — only runs in Colab.')